# NTU ML 2021 Spring — HW1: COVID-19 Cases Prediction

**任務**：用 DNN (Deep Neural Network) 做 Regression，根據美國某州過去 3 天的調查資料，預測第 3 天的新增陽性確診比例（`tested_positive`）。

**評估指標**：RMSE (Root Mean Squared Error)

---

> **Reference**
> Source: Heng-Jui Chang @ NTUEE
> https://github.com/ga642381/ML2021-Spring/blob/main/HW01/HW01.ipynb

## Cell 1：下載資料

In [ ]:
# 安裝 gdown 套件，用來從 Google Drive 直接下載檔案
# Colab 預設有 gdown，但版本可能舊，加上 -q 讓輸出簡潔
!pip install -q gdown

import os

# Google Drive 上的檔案 ID（從作業提供的連結取得）
TRAIN_FILE_ID = '19CCyCgJrUxtvgZF53vnctJiOJ23T5mqF'
TEST_FILE_ID  = '1CE240jY9XMquxW4-LcelkCBMMchN3f0R'

# 下載訓練資料（如果尚未下載）
# os.path.exists 先檢查避免重複下載
if not os.path.exists('covid.train.csv'):
    !gdown --id {TRAIN_FILE_ID} -O covid.train.csv
    print('covid.train.csv 下載完成')
else:
    print('covid.train.csv 已存在，跳過下載')

# 下載測試資料
if not os.path.exists('covid.test.csv'):
    !gdown --id {TEST_FILE_ID} -O covid.test.csv
    print('covid.test.csv 下載完成')
else:
    print('covid.test.csv 已存在，跳過下載')

# ============================================================
# 備用方式（如果 Google Drive 連結失效）：
# 請到 Kaggle 手動下載：
# https://www.kaggle.com/c/ml2021spring-hw1/data
# 下載後將 covid.train.csv 和 covid.test.csv 上傳到 Colab
# ============================================================

## Cell 2：Import 套件

In [ ]:
# PyTorch 核心套件
import torch
import torch.nn as nn                          # 神經網路模組（layers, activation functions, loss functions）
import torch.optim as optim                    # Optimizer（SGD, Adam 等）

# PyTorch 資料處理套件
from torch.utils.data import Dataset, DataLoader
# Dataset：定義「如何讀取一筆資料」的抽象類別
# DataLoader：自動幫我們做 batching、shuffle、多執行緒讀取

# 數值與資料處理
import numpy as np                             # 矩陣運算
import pandas as pd                            # 讀取 CSV、DataFrame 操作
import csv                                     # 存輸出結果
import os                                      # 檔案路徑操作
import math                                    # math.sqrt 用來算 RMSE

# 視覺化
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

print(f'PyTorch 版本：{torch.__version__}')
print(f'CUDA 是否可用：{torch.cuda.is_available()}')

## Cell 3：Utility Functions ＆ 超參數設定

In [ ]:
def same_seed(seed):
    """
    固定隨機種子，確保每次執行結果可重現（Reproducibility）。
    機器學習中很多地方會用到隨機數（參數初始化、資料 shuffle 等），
    固定 seed 後，相同 code 跑出來的結果就會一致。
    """
    # 固定 Python hash 隨機性（影響 set、dict 等）
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # 固定 numpy 的隨機種子
    np.random.seed(seed)

    # 固定 PyTorch CPU 的隨機種子
    torch.manual_seed(seed)

    # 若有 GPU，也固定 CUDA 的隨機種子
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_valid_split(data_set, valid_ratio, seed):
    """
    將訓練資料切分為 train set 和 validation set。
    valid_ratio：validation set 佔總資料的比例（例如 0.1 = 10%）
    用固定 seed shuffle 後切分，確保每次切出一樣的組合。
    """
    valid_set_size = int(valid_ratio * len(data_set))
    train_set_size = len(data_set) - valid_set_size

    # torch.utils.data.random_split 根據長度隨機切分
    train_set, valid_set = torch.utils.data.random_split(
        data_set,
        [train_set_size, valid_set_size],
        generator=torch.Generator().manual_seed(seed)
    )
    return train_set, valid_set


def predict(test_loader, model, device):
    """
    用訓練好的 model 對 test set 做預測。
    model.eval() 切換到評估模式：關閉 Dropout、BatchNorm 使用 running stats，
    讓推論結果穩定。
    """
    model.eval()  # 切換到評估模式（和 training mode 的差別在 Dropout / BN 行為）
    preds = []

    for x in test_loader:
        x = x.to(device)  # 將資料移到 GPU（或 CPU）

        # torch.no_grad()：推論時不需要計算 Gradient，節省記憶體與計算量
        with torch.no_grad():
            pred = model(x)

        # 將結果從 GPU 移回 CPU，轉成 numpy array 存起來
        preds.append(pred.detach().cpu())

    # 將所有 batch 的預測結果合併成一個大 tensor，再轉 numpy
    preds = torch.cat(preds, dim=0).numpy()
    return preds


# ============================================================
# 設定運算裝置：優先使用 GPU（CUDA），沒有 GPU 就用 CPU
# Colab 免費版可以在「執行階段 → 變更執行階段類型」選 GPU
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用裝置：{device}')

# ============================================================
# 超參數設定（Hyperparameters）
# 集中放在 config dict 方便管理，調整時不用到處找
# ============================================================
config = {
    'seed':          5201314,   # 隨機種子，確保結果可重現
    'select_all':    False,     # False = Feature Selection（Medium Baseline）
                                # True  = 使用全部 feature（Simple Baseline）
    'valid_ratio':   0.1,       # 10% 的訓練資料作為 Validation Set
    'n_epochs':      3000,      # 最多訓練幾個 Epoch
    'batch_size':    256,       # 每次 forward 傳入多少筆資料（越大 GPU 利用率越高，但需要更多記憶體）
    'learning_rate': 1e-5,      # SGD 的 Learning Rate（步伐大小）
    'early_stop':    400,       # 若 Validation Loss 連續 400 個 epoch 沒改善就提前停止
    'save_path':     './models/model.ckpt'  # 最佳模型的儲存路徑
}

## Cell 4：定義 Dataset

In [ ]:
class COVID19Dataset(Dataset):
    """
    繼承 PyTorch 的 Dataset 抽象類別，定義如何讀取 COVID-19 資料。

    繼承 Dataset 需要實作三個方法：
    - __init__：初始化，讀進資料、做前處理
    - __len__：回傳資料筆數（DataLoader 需要知道）
    - __getitem__：回傳第 i 筆資料（DataLoader 在 batching 時呼叫）
    """

    def __init__(self, path, mode='train', target_only=False):
        """
        path        : CSV 檔案路徑
        mode        : 'train' / 'valid' / 'test'
                      train 和 valid 都從 covid.train.csv 讀取，再用 random_split 切分
                      test  從 covid.test.csv 讀取（沒有 label）
        target_only : True  = Feature Selection，只選 states + tested_positive
                      False = 使用全部 93/94 個 feature
        """
        self.mode = mode

        # 用 pandas 讀取 CSV，取出 numpy array 方便後續操作
        # header=0 表示第一行是欄位名稱，iloc[:, 1:] 跳過第一欄（index 欄）
        with open(path, 'r') as fp:
            data = list(csv.reader(fp))
            data = np.array(data[1:])[:, 1:].astype(float)
            # data 的 shape：(筆數, 欄位數)
            # train: (2700, 94)，最後一欄是 label（tested_positive）
            # test:  (893,  93)，沒有最後一欄

        # ============================================================
        # Feature Selection
        # ============================================================
        # Feature 結構（共 93 個 feature + 1 label）：
        #   [0:40]   : 40 個州的 One-Hot Encoding
        #   [40:58]  : Day1 的 18 個 feature（最後一個 [57] 是 Day1 tested_positive）
        #   [58:76]  : Day2 的 18 個 feature（最後一個 [75] 是 Day2 tested_positive）
        #   [76:94]  : Day3 的 18 個 feature（最後一個 [93] 是 Day3 tested_positive = label）
        # target_only = True 時，只保留：
        #   states (0:40) + Day1 tested_positive (57) + Day2 tested_positive (75)
        #   + Day3 features（test 無 label；train/valid 有 label 在最後）

        if target_only:
            # 選取 states one-hot（前 40 欄）+ 每天的 tested_positive
            # Day1 tested_positive: index 57, Day2: index 75
            # 注意：index 93 是 label，不算在 feature 裡（test 沒有這欄）
            feats = list(range(40)) + [57, 75]
        else:
            # 用全部 feature（93 欄，不含 label）
            feats = list(range(93))

        if mode == 'test':
            # Test set：只有 feature，沒有 label
            self.data = torch.FloatTensor(data[:, feats])
        else:
            # Train / Valid：最後一欄（index 93）是 label
            self.data = torch.FloatTensor(data[:, feats])
            self.label = torch.FloatTensor(data[:, -1])  # 最後一欄 = Day3 tested_positive

        # 印出資料維度，方便確認讀取正確
        print(f'[{mode}] 資料筆數：{len(self.data)}，Feature 維度：{self.data.shape[1]}')

    def __len__(self):
        """回傳資料集的總筆數，DataLoader 需要這個資訊來決定 batch 的數量。"""
        return len(self.data)

    def __getitem__(self, idx):
        """
        回傳第 idx 筆資料。
        DataLoader 會呼叫這個方法來取得每筆資料，再組成 batch。
        """
        if self.mode in ['train', 'valid']:
            # 訓練和驗證模式：同時回傳 feature 和 label
            return self.data[idx], self.label[idx]
        else:
            # 測試模式：只回傳 feature（沒有 label）
            return self.data[idx]

## Cell 5：定義 DataLoader

In [ ]:
def prep_dataloader(path, mode, batch_size, n_jobs=0, target_only=False):
    """
    準備 DataLoader：讀取資料 → 建立 Dataset → 包裝成 DataLoader。

    DataLoader 的作用：
    1. 自動將資料切成一個一個 batch（每次 forward 傳入 batch_size 筆）
    2. 訓練時自動 shuffle，讓模型每個 epoch 看到不同順序的資料（避免過擬合）
    3. n_jobs > 0 時使用多執行緒預先讀取資料，加速訓練

    path        : CSV 檔案路徑
    mode        : 'train' / 'valid' / 'test'
    batch_size  : 每個 batch 的資料筆數
    n_jobs      : DataLoader 的 num_workers（Colab 建議用 0）
    target_only : 是否只用 Feature Selection 後的 feature
    """
    dataset = COVID19Dataset(path, mode=mode, target_only=target_only)

    # 訓練集要 shuffle，讓每個 epoch 的資料順序不同
    # 驗證集和測試集不 shuffle，結果才能對應到正確的 id
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(mode == 'train'),  # 只有 train mode 才 shuffle
        drop_last=False,            # 最後一個不完整的 batch 也保留
        num_workers=n_jobs,         # 讀取資料的並行 worker 數
        pin_memory=True             # 加速 CPU→GPU 的資料傳輸
    )
    return dataloader

## Cell 6：定義 DNN Model

In [ ]:
class NeuralNet(nn.Module):
    """
    三層全連接神經網路（Fully-Connected DNN）用於 Regression。

    繼承 nn.Module 是 PyTorch 定義神經網路的標準方式。
    需要實作：
    - __init__：定義網路架構（各層的 Layer）
    - forward：定義前向傳播（資料如何流過各層）
    """

    def __init__(self, input_dim):
        """
        input_dim : 輸入 feature 的維度（由 Feature Selection 決定）

        網路架構：
        Input(input_dim) → Linear → ReLU → Linear → ReLU → Linear → Output(1)
        隱藏層維度用 64，輸出維度是 1（預測一個數值：tested_positive 比例）
        """
        super(NeuralNet, self).__init__()

        # nn.Sequential 讓我們可以像堆積木一樣依序定義各層
        # 資料會從第一層依序流到最後一層
        self.net = nn.Sequential(
            # 第一層：input_dim → 64
            nn.Linear(input_dim, 64),
            nn.ReLU(),                  # Activation Function，引入非線性，讓網路能學習複雜的關係

            # 第二層：64 → 64
            nn.Linear(64, 64),
            nn.ReLU(),

            # 第三層（輸出層）：64 → 1
            # Regression 輸出層不加 Activation Function，直接輸出連續數值
            nn.Linear(64, 1)
        )

        # 設定 Loss Function 為 MSELoss（Mean Squared Error）
        # Regression 問題的標準 Loss Function：預測值和真實值差的平方的平均
        self.criterion = nn.MSELoss(reduction='mean')

    def forward(self, x):
        """
        前向傳播：定義資料 x 如何流過網路。
        x 的 shape：(batch_size, input_dim)
        """
        # 將 x 傳入 Sequential，輸出 shape：(batch_size, 1)
        # squeeze(1) 將維度 1 的那個軸壓縮掉，變成 (batch_size,)
        # 這樣才能和 label（shape: (batch_size,)）做 MSELoss 計算
        return self.net(x).squeeze(1)

    def cal_loss(self, pred, target):
        """
        計算 MSE Loss。
        之所以獨立成一個 method，是方便未來加入 Regularization（如 L2 正則化）。
        """
        return self.criterion(pred, target)

## Cell 7：Training Loop

In [ ]:
def trainer(train_loader, valid_loader, model, config, device):
    """
    訓練流程：
    1. 每個 epoch 跑完所有 training batch → 計算 train loss
    2. 用 validation set 評估目前模型 → 計算 valid loss
    3. 若 valid loss 創新低，儲存模型
    4. 若連續 early_stop 個 epoch valid loss 沒有改善，提前停止

    回傳 train_loss_record 和 valid_loss_record（畫圖用）
    """

    # 使用 SGD Optimizer（隨機梯度下降法）
    # momentum=0.9 讓更新方向有「慣性」，加速收斂並跳脫局部最小值
    optimizer = optim.SGD(
        model.parameters(),
        lr=config['learning_rate'],
        momentum=0.9
    )

    # 確保模型儲存路徑的資料夾存在
    os.makedirs(os.path.dirname(config['save_path']), exist_ok=True)

    # Early Stopping 相關變數
    n_epochs         = config['n_epochs']
    best_loss        = math.inf    # 記錄歷史最佳 valid loss（初始設定為無限大）
    step             = 0           # 記錄總 training step 數（每個 batch 算一步）
    early_stop_count = 0           # 連續幾個 epoch valid loss 沒有改善的計數器

    # 記錄每個 epoch 的 loss（畫圖用）
    train_loss_record = []
    valid_loss_record = []

    for epoch in range(n_epochs):

        # ---- Training Phase ----
        model.train()  # 切換到訓練模式（啟用 Dropout / BatchNorm 的訓練行為）
        train_loss_sum  = 0
        train_batch_cnt = 0

        for x_batch, y_batch in train_loader:
            # 將資料移到運算裝置（GPU / CPU）
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward Pass：將 x 送進模型，得到預測值
            pred = model(x_batch)

            # 計算 Loss（MSE：(預測值 - 真實值)^2 的平均）
            loss = model.cal_loss(pred, y_batch)

            # Backward Pass：計算每個參數的 Gradient（偏微分）
            # 必須先清零上一步的 Gradient，否則會累加
            optimizer.zero_grad()
            loss.backward()

            # 更新參數：用 Gradient 和 Learning Rate 更新每個參數
            # 參數 = 參數 - learning_rate × gradient（Gradient Descent）
            optimizer.step()

            train_loss_sum  += loss.item()  # .item() 從 tensor 取出 Python float
            train_batch_cnt += 1
            step += 1

        # 計算這個 epoch 的平均 train loss
        avg_train_loss = train_loss_sum / train_batch_cnt
        train_loss_record.append(avg_train_loss)

        # ---- Validation Phase ----
        # 每個 epoch 結束後用 validation set 評估模型效能
        model.eval()  # 切換到評估模式
        valid_loss_sum  = 0
        valid_batch_cnt = 0

        with torch.no_grad():  # 驗證時不需要算 Gradient，節省記憶體
            for x_val, y_val in valid_loader:
                x_val = x_val.to(device)
                y_val = y_val.to(device)

                pred_val   = model(x_val)
                loss_val   = model.cal_loss(pred_val, y_val)
                valid_loss_sum  += loss_val.item()
                valid_batch_cnt += 1

        avg_valid_loss = valid_loss_sum / valid_batch_cnt
        valid_loss_record.append(avg_valid_loss)

        # 每 100 個 epoch 印一次進度
        if (epoch + 1) % 100 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1:4d}/{n_epochs}] '
                  f'Train Loss: {avg_train_loss:.4f}  '
                  f'Valid Loss: {avg_valid_loss:.4f}  '
                  f'(RMSE: {math.sqrt(avg_valid_loss):.4f})')

        # ---- 儲存最佳模型 & Early Stopping ----
        if avg_valid_loss < best_loss:
            # Valid loss 創新低：儲存模型並重置計數器
            best_loss = avg_valid_loss
            torch.save(model.state_dict(), config['save_path'])
            # state_dict() 取出模型所有參數（weights & biases）的字典
            early_stop_count = 0
        else:
            # Valid loss 沒有改善：計數器 +1
            early_stop_count += 1

        if early_stop_count >= config['early_stop']:
            print(f'\nEarly Stopping 觸發！已連續 {config["early_stop"]} 個 epoch Valid Loss 沒有改善。')
            print(f'最佳 Valid Loss: {best_loss:.4f}，對應 RMSE: {math.sqrt(best_loss):.4f}')
            break

    print(f'\n訓練完成！最佳模型已存至 {config["save_path"]}')
    return train_loss_record, valid_loss_record

## Cell 8：畫出 Loss 曲線

In [ ]:
def plot_learning_curve(train_loss_record, valid_loss_record, title=''):
    """
    畫出 Training Loss 和 Validation Loss 的曲線。

    這張圖可以幫助我們診斷訓練狀況：
    - 兩條曲線都高：Underfitting（模型太簡單，或 epoch 不夠多）
    - Train loss 低但 Valid loss 高：Overfitting（模型記住訓練資料，泛化能力差）
    - 兩條曲線都低且接近：理想狀態
    """
    total_steps = len(train_loss_record)
    x_1 = range(total_steps)

    figure(figsize=(10, 4))

    plt.plot(x_1, train_loss_record, label='Train Loss', color='blue',  linewidth=1)
    plt.plot(x_1, valid_loss_record, label='Valid Loss', color='orange', linewidth=1)

    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title(f'Learning Curve — {title}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print('Loss 曲線圖繪製完成。')

## Cell 9：產生預測結果

In [ ]:
def save_pred(preds, file):
    """
    將預測結果存成 CSV，格式符合 Kaggle 提交要求：
    id,tested_positive
    0,12.34
    1,56.78
    ...
    """
    print(f'儲存預測結果至 {file}...')
    with open(file, 'w') as fp:
        writer = csv.writer(fp)
        writer.writerow(['id', 'tested_positive'])  # 表頭
        for i, p in enumerate(preds):
            writer.writerow([i, p])
    print(f'完成！共 {len(preds)} 筆預測結果。')

## Cell 10：執行完整訓練流程

In [ ]:
# ============================================================
# Step 1：固定隨機種子，確保結果可重現
# ============================================================
same_seed(config['seed'])

# ============================================================
# Step 2：準備 DataLoader
# ============================================================
print('=== 載入資料 ===')

# 先建立完整的訓練 Dataset，再用 random_split 切出 valid set
# 這樣 train 和 valid 是從同一個 CSV 切出來的，不會有資料洩漏
train_dataset = COVID19Dataset('covid.train.csv', mode='train',
                               target_only=config['select_all'] is False)
test_dataset  = COVID19Dataset('covid.test.csv',  mode='test',
                               target_only=config['select_all'] is False)

# 切分 train / valid（90% / 10%）
train_set, valid_set = train_valid_split(
    train_dataset,
    config['valid_ratio'],
    config['seed']
)

print(f'Train set 大小：{len(train_set)}，Valid set 大小：{len(valid_set)}，Test set 大小：{len(test_dataset)}')

# 將 Dataset 包裝成 DataLoader
# n_jobs=0：在 Colab 上用單執行緒讀取（多執行緒在 Colab 有時會有問題）
train_loader = DataLoader(train_set,     batch_size=config['batch_size'], shuffle=True,  drop_last=False, pin_memory=True, num_workers=0)
valid_loader = DataLoader(valid_set,     batch_size=config['batch_size'], shuffle=False, drop_last=False, pin_memory=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=config['batch_size'], shuffle=False, drop_last=False, pin_memory=True, num_workers=0)

# ============================================================
# Step 3：建立模型
# ============================================================
print('\n=== 建立模型 ===')

# 取得 feature 維度（從 dataset 的第一筆資料得知）
# train_set[0][0] 是第一筆資料的 feature tensor，.shape[0] 是維度
input_dim = train_set[0][0].shape[0]
print(f'Feature 維度：{input_dim}')

model = NeuralNet(input_dim).to(device)  # 將模型移到 GPU/CPU
print(model)  # 印出模型架構

# 計算模型參數量
total_params = sum(p.numel() for p in model.parameters())
print(f'模型總參數數量：{total_params:,}')

# ============================================================
# Step 4：開始訓練
# ============================================================
print('\n=== 開始訓練 ===')
train_loss_record, valid_loss_record = trainer(
    train_loader, valid_loader, model, config, device
)

# ============================================================
# Step 5：畫 Loss 曲線
# ============================================================
print('\n=== 繪製 Loss 曲線 ===')
plot_learning_curve(
    train_loss_record,
    valid_loss_record,
    title='COVID-19 Cases Prediction'
)

# ============================================================
# Step 6：載入最佳模型，對 Test set 做預測，存成 CSV
# ============================================================
print('\n=== 產生預測結果 ===')

# 載入訓練過程中儲存的最佳模型參數
model.load_state_dict(torch.load(config['save_path']))

preds = predict(test_loader, model, device)
save_pred(preds, 'pred.csv')

print('\n全部完成！請將 pred.csv 提交至 Kaggle。')